# Audit: what does `tabpfn` do by default?

**CPU only. No model weights downloaded. Runs in seconds.**

This notebook inspects the *installed* `tabpfn` package - signatures, default
values, docstrings, constants - and prints file paths and line numbers so that
every claim in the README can be checked against the source.

**What to look at.** Section 2 prints the defaults of `FinetunedTabPFNClassifier`.
The three that matter:

* `n_finetune_ctx_plus_query_samples` - a cap on context + query per episode
* `finetune_ctx_query_split_ratio` - the query share carved out of it
* `n_inference_subsample_samples` - `None` means the full training set is context

Together they lock the ratio at **1.25** for any training set below the cap, and
release it above. Section 5 classifies the finding automatically.

*Output is printed with file paths and line numbers so that every claim in the
README can be checked against the installed source.*

---


## What the installed package actually does

**No GPU. No weights downloaded. A few seconds.** This notebook only introspects
the installed package: signatures, default values, docstrings and constants. It
instantiates no model, so it can run while another notebook holds the GPU quota.
Set the accelerator to **None**.

**The question.** Measurements over 24 datasets show that the gain from
fine-tuning depends on the ratio between the training episode length and the
inference context length. What remains is to establish **what the library does
when the user specifies nothing.**

Three outcomes are possible, and they do not carry the same weight:

| What we find | What it means |
|:--|:--|
| a **constant** default (1024, 2048, ...) independent of the data | the default user is off the diagonal: a tuning defect, with measurements behind it |
| a default that takes the **whole training set** | the library is on the diagonal by default; the result concerns only those who set this parameter by hand |
| the parameter is **not exposed** | the behaviour is imposed; it remains to be seen which one |

The output of this notebook is citable as it stands: it prints the version, the
file paths and the line numbers.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0'],check=True)
print('tabpfn installed.')

## 1. Version and location of the package

In [ ]:

import importlib.metadata, pathlib, inspect, re, textwrap, json
import tabpfn

VERSION = importlib.metadata.version('tabpfn')
ROOT = pathlib.Path(tabpfn.__file__).parent
print('tabpfn        :', VERSION)
print('location      :', ROOT)
print('submodules    :', sorted(p.name for p in ROOT.iterdir() if p.is_dir() and not p.name.startswith('_')))

def show(obj, name):
    try:
        print(f'\n--- {name} ---')
        print(inspect.signature(obj))
    except Exception as exc:
        print(f'\n--- {name}: unavailable ({type(exc).__name__}: {exc}) ---')

def doc(obj, name, n=2500):
    d = inspect.getdoc(obj) or '(no docstring)'
    print(f'\n===== docstring: {name} =====')
    print(d[:n] + ('…' if len(d) > n else ''))

## 2. The fine-tuning API

Signatures and default values. This is where the answer lies: is the training
support size a constant, or does it depend on the data?

In [ ]:

TARGETS = []
try:
    from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
    TARGETS.append((FinetunedTabPFNClassifier.__init__,'FinetunedTabPFNClassifier.__init__'))
    TARGETS.append((FinetunedTabPFNClassifier.fit,'FinetunedTabPFNClassifier.fit'))
except Exception as exc:
    print('FinetunedTabPFNClassifier not found:',type(exc).__name__,exc)
    FinetunedTabPFNClassifier=None
try:
    from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
    TARGETS.append((get_preprocessed_dataset_chunks,'get_preprocessed_dataset_chunks'))
except Exception as exc:
    print('data_util not found:',type(exc).__name__,exc)
    get_preprocessed_dataset_chunks=None

for obj,name in TARGETS: show(obj,name)

print('\n\n==================== DEFAULT VALUES ====================')
for obj,name in TARGETS:
    try: sig=inspect.signature(obj)
    except Exception: continue
    print(f'\n{name}')
    for p in sig.parameters.values():
        if p.name in ('self','args','kwargs'): continue
        d = '(required)' if p.default is inspect._empty else repr(p.default)
        print(f'   {p.name:<34} = {d}')

if FinetunedTabPFNClassifier is not None:
    doc(FinetunedTabPFNClassifier,'FinetunedTabPFNClassifier')
if get_preprocessed_dataset_chunks is not None:
    doc(get_preprocessed_dataset_chunks,'get_preprocessed_dataset_chunks')

## 3. Sweeping the source

Everything that looks like a context, support or batch size, with its file and
line number, so that the claim can be cited.

In [ ]:

PAT = re.compile(r'(max_data_size|max_samples|support|context_size|n_context|chunk_size'
                 r'|max_context|n_samples_|MAX_|LIMIT|subsample)')
def sweep(folder,title):
    d = ROOT/folder if folder else ROOT
    if not d.exists(): print(f'\n{title}: folder missing'); return
    print(f'\n=================== {title} ===================')
    for p in sorted(d.rglob('*.py')):
        rows=[]
        for i,line in enumerate(p.read_text(errors='replace').splitlines(),1):
            s=line.strip()
            if not s or s.startswith('#'): continue
            if PAT.search(s) and ('=' in s or 'def ' in s):
                rows.append((i,s))
        if rows:
            print(f'\n  {p.relative_to(ROOT)}')
            for i,s in rows[:40]:
                print(f'     {i:>5} | {s[:118]}')
            if len(rows)>40: print(f'     ... {len(rows)-40} more lines')

sweep('finetuning','FINETUNING MODULE')
sweep('constants.py' if (ROOT/'constants.py').exists() else '','CONSTANTS (root)')

## 4. The inference side

By symmetry: what does `TabPFNClassifier` do when handed more rows than its
pre-training limits? That is the other half of the ratio.

In [ ]:

try:
    from tabpfn import TabPFNClassifier
    show(TabPFNClassifier.__init__,'TabPFNClassifier.__init__')
    sig=inspect.signature(TabPFNClassifier.__init__)
    print('\nrelevant default values:')
    for name in ['n_estimators','fit_mode','memory_saving_mode','ignore_pretraining_limits',
                'inference_config','random_state','device']:
        if name in sig.parameters:
            p=sig.parameters[name]
            print(f'   {name:<28} = {"(required)" if p.default is inspect._empty else repr(p.default)}')
    d=inspect.getdoc(TabPFNClassifier) or ''
    for term in ['ignore_pretraining_limits','pretraining','context','subsample','10000','memory']:
        for para in d.split('\n\n'):
            if term in para:
                print(f'\n--- docstring, mentions \'{term}\' ---')
                print(textwrap.fill(para.strip()[:700],100)); break
except Exception as exc:
    print('TabPFNClassifier:',type(exc).__name__,exc)

## 5. Verdict

Automatic classification into one of the three outcomes. Read it with the output
of the previous sections in view: the verdict is an aid, not a proof.

In [ ]:

verdict=[]
def default_of(obj,names):
    try: sig=inspect.signature(obj)
    except Exception: return None
    for n in names:
        if n in sig.parameters:
            p=sig.parameters[n]
            return (n, None if p.default is inspect._empty else p.default)
    return None

CANDIDATES=['max_data_size','max_samples','support_size','context_size','max_context_size',
           'n_context','chunk_size']
hits=[]
for obj,name in TARGETS:
    d=default_of(obj,CANDIDATES)
    if d: hits.append((name,)+d)

print('size parameters exposed:')
if not hits:
    print('   NONE - the parameter is not exposed in these signatures.')
    verdict.append("CASE 3: support size is not a user parameter of this API. "
                   "See section 3 for what the code actually enforces.")
for name,param,val in hits:
    print(f'   {name}.{param} = {val!r}')
    if val is None:
        verdict.append(f"CASE 2 (weak): {name}.{param} defaults to None - check in "
                       "section 3, but it most likely means 'use the whole "
                       "training set', so the library is on the diagonal by default.")
    elif isinstance(val,(int,float)):
        verdict.append(f"CASE 1 (strong): {name}.{param} defaults to {val}, a constant "
                       "independent of the data. A user who fine-tunes without specifying "
                       f"anything and then predicts with more than {val} context rows is off "
                       "diagonal without having chosen to be.")
    else:
        verdict.append(f"To inspect by hand: {name}.{param} = {val!r}")

print('\n==================== VERDICT ====================')
for v in verdict or ["Nothing conclusive automatically: read sections 2 and 3."]:
    print('\n* '+textwrap.fill(v,96).replace('\n','\n  '))
print('\nVersion inspected: tabpfn',VERSION)
print('Cite with the file and line number printed in section 3.')

## What to make of it

**If it is case 1** — a constant default — that is the most useful finding here:
it turns an empirical observation into a reproducible tuning defect, with a
measure of its cost. The fix is one line: make the fine-tuning support size
depend on the expected inference context, or draw it at random over the
deployment range.

**If it is case 2** — the default takes the whole training set — the library
already does the right thing. The result stays true but concerns only those who
set this parameter by hand, and the write-up must say so rather than imply the
default is at fault.

**In both cases**, keep the full output: the version, the paths and the line
numbers are what make the claim checkable by a reviewer.